In [ ]:
#============================================================
# Experiment 11A: Random Forest Classifier from Scratch
# Synthetic Binary Classification Dataset (Student Performance)
#============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1: Create Synthetic Dataset
# X1 = Study Hours, X2 = Attendance
# Class 0 = Fail, Class 1 = Pass
X = np.array([
    [1, 55],
    [1.5, 58],
    [2, 60],
    [2.5, 63],
    [3, 65],
    [3.5, 67],
    [4, 70],
    [4.5, 72],
    [5, 75],
    [5.5, 78],
    [6, 80],
    [6.5, 82],
    [7, 85],
    [7.5, 88],
    [8, 90],
    [8.5, 92]
], dtype=float)

y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])

# Step 2: Train-Test Split From Scratch
np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.8 * len(X))

train_indices = indices[:train_size]
test_indices = indices[train_size:]

X_train = X[train_indices]
X_test = X[test_indices]
y_train = y[train_indices]
y_test = y[test_indices]

print("Training Samples:", len(X_train))
print("Testing Samples:", len(X_test))

# Step 3: Gini Impurity
def gini_impurity(y):
    if len(y) == 0:
        return 0
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    gini = 1 - np.sum(probabilities**2)
    return gini

# Step 4: Calculate Weighted Gini
def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    gini_left = gini_impurity(y_left)
    gini_right = gini_impurity(y_right)
    weighted = (len(y_left) / n) * gini_left + (len(y_right) / n) * gini_right
    return weighted

# Step 5: Split Dataset
def split_dataset(X, y, feature, threshold):
    left_mask = (X[:, feature] <= threshold)
    right_mask = ~left_mask

    X_left = X[left_mask]
    y_left = y[left_mask]
    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, y_left, X_right, y_right

# Step 6: Find Best Split
def best_split(X, y, feature_indices):
    best_gini = float("inf")
    best_feature = None
    best_threshold = None

    for feature in feature_indices:
        values = np.unique(X[:, feature])

        for threshold in values:
            X_left, y_left, X_right, y_right = split_dataset(X, y, feature, threshold)

            if len(y_left) == 0 or len(y_right) == 0:
                continue

            gini = weighted_gini(y_left, y_right)

            if gini < best_gini:
                best_gini = gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gini

# Step 7: Decision Tree Node
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# Step 8: Decision Tree Classifier
class DecisionTree:

    def __init__(self, max_depth=5, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None
        self.feature_importance = None

    # Training
    def fit(self, X, y):
        self.feature_importance = np.zeros(X.shape[1])
        self.root = self._grow_tree(X, y, depth=0)

    # Grow Tree
    def _grow_tree(self, X, y, depth):
        n_samples = len(y)
        n_classes = len(np.unique(y))

        # Stopping Conditions
        if (depth >= self.max_depth or
                n_classes == 1 or
                n_samples < self.min_samples_split):
            return Node(value=self._majority_class(y))

        # Random Feature Selection
        n_features = X.shape[1]
        if self.max_features is None:
            n_selected = int(np.sqrt(n_features))
        else:
            n_selected = self.max_features

        n_selected = max(1, min(n_selected, n_features))
        feature_indices = np.random.choice(n_features, n_selected, replace=False)

        # Find Best Split
        feature, threshold, gini = best_split(X, y, feature_indices)

        if feature is None:
            return Node(value=self._majority_class(y))

        # Split Dataset
        X_left, y_left, X_right, y_right = split_dataset(X, y, feature, threshold)

        # Feature Importance
        parent_gini = gini_impurity(y)
        n = len(y)
        weighted_child_gini = ((len(y_left) / n) * gini_impurity(y_left) +
                               (len(y_right) / n) * gini_impurity(y_right))
        improvement = parent_gini - weighted_child_gini
        self.feature_importance[feature] += improvement * n

        # Recursive Tree Growth
        left_child = self._grow_tree(X_left, y_left, depth + 1)
        right_child = self._grow_tree(X_right, y_right, depth + 1)

        return Node(feature=feature, threshold=threshold,
                    left=left_child, right=right_child)

    # Majority Class
    def _majority_class(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    # Predict One Sample
    def _predict_one(self, x, node):
        if node.value is not None:
            return node.value
        if x[node.feature] <= node.threshold:
            return self._predict_one(x, node.left)
        else:
            return self._predict_one(x, node.right)

    # Predict Multiple Samples
    def predict(self, X):
        return np.array([self._predict_one(x, self.root) for x in X])

# Step 9: Random Forest Classifier
class RandomForest:

    def __init__(self, n_trees=10, max_depth=5, min_samples_split=2):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    # Bootstrap Sampling (with replacement)
    def bootstrap_sample(self, X, y):
        n_samples = len(X)
        indices = np.random.choice(n_samples, n_samples, replace=True)
        return X[indices], y[indices]

    # Train Random Forest
    def fit(self, X, y):
        self.trees = []
        for i in range(self.n_trees):
            # Bootstrap dataset
            X_sample, y_sample = self.bootstrap_sample(X, y)

            # Create tree
            tree = DecisionTree(max_depth=self.max_depth,
                                min_samples_split=self.min_samples_split)
            # Train tree
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
            print("Tree", i + 1, "trained")

    # Predict using Majority Voting
    def predict(self, X):
        tree_predictions = np.array([tree.predict(X) for tree in self.trees])
        final_predictions = []

        for i in range(X.shape[0]):
            values, counts = np.unique(tree_predictions[:, i], return_counts=True)
            final_predictions.append(values[np.argmax(counts)])

        return np.array(final_predictions)

# Step 10: Create Random Forest
np.random.seed(42)
forest = RandomForest(n_trees=10, max_depth=5, min_samples_split=2)

# Step 11: Train Random Forest
print("\n===================================")
print("TRAINING RANDOM FOREST")
print("===================================")
forest.fit(X_train, y_train)

# Step 12: Make Predictions
y_pred = forest.predict(X_test)

# Step 13: Display Predictions
print("\nActual vs Predicted")
print("==============================")
print("Actual\tPredicted")
for i in range(len(y_test)):
    print(f"{y_test[i]}\t{y_pred[i]}")

# Step 14: Calculate Accuracy
accuracy = np.mean(y_pred == y_test)

print("\n===================================")
print("RANDOM FOREST PERFORMANCE")
print("===================================")
print("Correct Predictions:", np.sum(y_pred == y_test))
print("Total Predictions:", len(y_test))
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))

# Step 15: Confusion Matrix From Scratch
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == 0) & (y_pred == 0))
FP = np.sum((y_test == 0) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == 0))

print("\nConfusion Matrix")
print("-------------------------")
print("Predicted")
print("  0   1")
print(f"Actual 0 {TN:3d} {FP:3d}")
print(f"Actual 1 {FN:3d} {TP:3d}")

# Steps 16-18: Precision, Recall, F1 Score
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0
f1 = (2 * precision * recall / (precision + recall)
      if (precision + recall) != 0 else 0)

print("\nClassification Metrics")
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

# Step 19: Predict New Student
new_student = np.array([[5.0, 76]])
new_prediction = forest.predict(new_student)[0]

print("\n===================================")
print("NEW STUDENT PREDICTION")
print("===================================")
print("Study Hours:", new_student[0, 0])
print("Attendance:", new_student[0, 1])
print("Predicted Class:", new_prediction)
if new_prediction == 1:
    print("Result: PASS")
else:
    print("Result: FAIL")

# Step 20: Plot Dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == 0, 0], X[y == 0, 1], label="Fail", marker="o")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Pass", marker="x")
plt.scatter(new_student[0, 0], new_student[0, 1],
            marker="*", s=180, label="New Student")
plt.xlabel("Study Hours")
plt.ylabel("Attendance (%)")
plt.title("Random Forest Classification")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#============================================================
# Experiment 11B: Random Forest Classifier Using a Real-World Dataset
# Breast Cancer Dataset - Scikit-learn
#============================================================

# Step 1: Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score
)

# Step 2: Load the Real-World Dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target
feature_names = cancer.feature_names
target_names = cancer.target_names

# Step 3: Create a DataFrame
df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y

print("==========================================")
print("BREAST CANCER DATASET")
print("==========================================")
print("\nDataset Shape:")
print(df.shape)
print("\nFirst Five Records:")
print(df.head())
print("\nFeature Names:")
print(list(feature_names))
print("\nTarget Classes:")
print(target_names)
print("\nClass Distribution:")
print(df["Target"].value_counts())

# Step 4: Separate Features and Target
X = df[feature_names].values
y = df["Target"].values

# Step 5: Train-Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("\nTraining Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

# Step 6: Create Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

# Step 7: Train the Model
rf_model.fit(X_train, y_train)
print("\nRandom Forest Model Training Completed Successfully.")

# Step 8: Display Model Information
print("\n==========================================")
print("RANDOM FOREST MODEL INFORMATION")
print("==========================================")
print("Number of Trees:")
print(rf_model.n_estimators)
print("\nNumber of Features:")
print(rf_model.n_features_in_)
print("\nClasses:")
print(rf_model.classes_)

# Step 9: Make Predictions
y_pred = rf_model.predict(X_test)

# Step 10: Prediction Probabilities
y_probability = rf_model.predict_proba(X_test)

# Step 11: Actual vs Predicted Results
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability_Malignant": y_probability[:, 0],
    "Probability_Benign": y_probability[:, 1]
})

print("\n==========================================")
print("ACTUAL VS PREDICTED")
print("==========================================")
print(results.head(20))

# Steps 12-15: Accuracy, Precision, Recall, F1
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

# Step 16: Display Performance
print("\n==========================================")
print("RANDOM FOREST CLASSIFICATION PERFORMANCE")
print("==========================================")
print("Accuracy:", round(accuracy, 4))
print("Accuracy (%):", round(accuracy * 100, 2))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("F1 Score:", round(f1, 4))

# Step 17: Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n==========================================")
print("CONFUSION MATRIX")
print("==========================================")
print(cm)

# Step 18: Classification Report
print("\n==========================================")
print("CLASSIFICATION REPORT")
print("==========================================")
print(classification_report(y_test, y_pred, target_names=target_names))

# Step 19: Feature Importance
feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": rf_model.feature_importances_
})
feature_importance = feature_importance.sort_values(by="Importance", ascending=False)

print("\n==========================================")
print("TOP FEATURE IMPORTANCE")
print("==========================================")
print(feature_importance.head(15))

# Step 20: Plot Confusion Matrix
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Random Forest Confusion Matrix")
plt.colorbar()
tick_marks = np.arange(len(target_names))
plt.xticks(tick_marks, target_names)
plt.yticks(tick_marks, target_names)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")
plt.tight_layout()
plt.show()

# Step 21: Plot Feature Importance
top_features = feature_importance.head(15)

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Top 15 Important Features - Random Forest")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Step 22: Actual vs Predicted Visualization
number_of_samples = min(30, len(y_test))
sample_index = np.arange(number_of_samples)

plt.figure(figsize=(10, 5))
plt.scatter(sample_index, y_test[:number_of_samples], marker="o", label="Actual")
plt.scatter(sample_index, y_pred[:number_of_samples], marker="x", label="Predicted")
plt.xlabel("Test Sample Index")
plt.ylabel("Class")
plt.title("Random Forest: Actual vs Predicted")
plt.yticks([0, 1], target_names)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Step 23: Display Individual Tree Depths
tree_depths = []
for tree in rf_model.estimators_:
    tree_depths.append(tree.get_depth())

print("\n==========================================")
print("INDIVIDUAL TREE INFORMATION")
print("==========================================")
print("Tree Depths:", tree_depths)
print("Average Tree Depth:", round(np.mean(tree_depths), 2))

# Step 24: Predict a New Sample
new_sample = X_test[0].reshape(1, -1)
new_prediction = rf_model.predict(new_sample)
new_probability = rf_model.predict_proba(new_sample)

print("\n==========================================")
print("NEW SAMPLE PREDICTION")
print("==========================================")
print("Predicted Class:", target_names[new_prediction[0]])
print("\nPrediction Probabilities:")
for i in range(len(target_names)):
    print(f"{target_names[i]}: {new_probability[0][i]:.4f}")

# Step 25: Final Result
print("\n==========================================")
print("FINAL RESULT")
print("==========================================")
print("Random Forest was successfully implemented"
      " using the Breast Cancer dataset available"
      " directly from Scikit-learn.")